In [2]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, types

# 1. SETUP YOUR CONNECTION (Using your server name from the screenshot)
server_name = 'localhost\\SQLEXPRESS'
database_name = 'OlistDB'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server_name};"
    f"DATABASE={database_name};"
    f"Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 2. POINT TO YOUR FOLDER (Your exact path)
folder_path = r'C:\Program Files\Microsoft SQL Server\MSSQL17.SQLEXPRESS\MSSQL\DATA\OlistCSV'

# 3. EXACT DATA TYPES
# 3. EXACT DATA TYPES (Fixed SQLAlchemy date types)
dtype_mapping = {
    'olist_customers_dataset.csv': {
        'customer_id': types.NVARCHAR(50), 'customer_unique_id': types.NVARCHAR(50),
        'customer_zip_code_prefix': types.NVARCHAR(10), 'customer_city': types.NVARCHAR(100), 'customer_state': types.NVARCHAR(10)
    },
    'olist_geolocation_dataset.csv': {
        'geolocation_zip_code_prefix': types.NVARCHAR(10), 'geolocation_lat': types.FLOAT(),
        'geolocation_lng': types.FLOAT(), 'geolocation_city': types.NVARCHAR(100), 'geolocation_state': types.NVARCHAR(10)
    },
    'olist_orders_dataset.csv': {
        'order_id': types.NVARCHAR(50), 'customer_id': types.NVARCHAR(50), 'order_status': types.NVARCHAR(20),
        'order_purchase_timestamp': types.DateTime(), 'order_approved_at': types.DateTime(),
        'order_delivered_carrier_date': types.DateTime(), 'order_delivered_customer_date': types.DateTime(), 'order_estimated_delivery_date': types.DateTime()
    },
    'olist_order_items_dataset.csv': {
        'order_id': types.NVARCHAR(50), 'order_item_id': types.INT(), 'product_id': types.NVARCHAR(50),
        'seller_id': types.NVARCHAR(50), 'shipping_limit_date': types.DateTime(), 'price': types.DECIMAL(10,2), 'freight_value': types.DECIMAL(10,2)
    },
    'olist_order_payments_dataset.csv': {
        'order_id': types.NVARCHAR(50), 'payment_sequential': types.INT(), 'payment_type': types.NVARCHAR(30),
        'payment_installments': types.INT(), 'payment_value': types.DECIMAL(10,2)
    },
    'olist_order_reviews_dataset.csv': {
        'review_id': types.NVARCHAR(50), 'order_id': types.NVARCHAR(50), 'review_score': types.INT(),
        'review_comment_title': types.NVARCHAR(255), 'review_comment_message': types.NVARCHAR(1000),
        'review_creation_date': types.DateTime(), 'review_answer_timestamp': types.DateTime()
    },
    'olist_products_dataset.csv': {
        'product_id': types.NVARCHAR(50), 'product_category_name': types.NVARCHAR(100),
        'product_name_lenght': types.FLOAT(), 'product_description_lenght': types.FLOAT(), 'product_photos_qty': types.FLOAT(),
        'product_weight_g': types.FLOAT(), 'product_length_cm': types.FLOAT(), 'product_height_cm': types.FLOAT(), 'product_width_cm': types.FLOAT()
    },
    'olist_sellers_dataset.csv': {
        'seller_id': types.NVARCHAR(50), 'seller_zip_code_prefix': types.NVARCHAR(10), 'seller_city': types.NVARCHAR(100), 'seller_state': types.NVARCHAR(10)
    },
    'product_category_name_translation.csv': {
        'product_category_name': types.NVARCHAR(100), 'product_category_name_english': types.NVARCHAR(100)
    }
}

# 4. FILES MAPPING (Leave this exactly as is)
files_to_load = {
    'olist_customers_dataset.csv': 'bronze_olist_customers',
    'olist_geolocation_dataset.csv': 'bronze_olist_geolocation',
    'olist_orders_dataset.csv': 'bronze_olist_orders',
    'olist_order_items_dataset.csv': 'bronze_olist_order_items',
    'olist_order_payments_dataset.csv': 'bronze_olist_order_payments',
    'olist_order_reviews_dataset.csv': 'bronze_olist_order_reviews',
    'olist_products_dataset.csv': 'bronze_olist_products',
    'olist_sellers_dataset.csv': 'bronze_olist_sellers',
    'product_category_name_translation.csv': 'bronze_product_category_name_translation'
}

# 5. LOOP AND LOAD
for filename, table_name in files_to_load.items():
    print(f"Loading {filename}...")
    try:
        df = pd.read_csv(f"{folder_path}\\{filename}", encoding='utf-8')
        
        df.to_sql(
            table_name, 
            con=engine, 
            if_exists='replace', 
            index=False, 
            dtype=dtype_mapping[filename]
        )
        
        print(f"✅ Success: {len(df)} rows loaded into {table_name}\n")
    except Exception as e:
        print(f"❌ Failed on {filename}: {e}")

print("🚀 ALL DONE! Go check SSMS.")

Loading olist_customers_dataset.csv...
✅ Success: 99441 rows loaded into bronze_olist_customers

Loading olist_geolocation_dataset.csv...
✅ Success: 1000163 rows loaded into bronze_olist_geolocation

Loading olist_orders_dataset.csv...
✅ Success: 99441 rows loaded into bronze_olist_orders

Loading olist_order_items_dataset.csv...
✅ Success: 112650 rows loaded into bronze_olist_order_items

Loading olist_order_payments_dataset.csv...
✅ Success: 103886 rows loaded into bronze_olist_order_payments

Loading olist_order_reviews_dataset.csv...
✅ Success: 99224 rows loaded into bronze_olist_order_reviews

Loading olist_products_dataset.csv...
✅ Success: 32951 rows loaded into bronze_olist_products

Loading olist_sellers_dataset.csv...
✅ Success: 3095 rows loaded into bronze_olist_sellers

Loading product_category_name_translation.csv...
✅ Success: 71 rows loaded into bronze_product_category_name_translation

🚀 ALL DONE! Go check SSMS.
